# NUMPY MASTERY — CHAPTERS 30–32

```
NUMPY MASTERY
Current Chapters: 30 (Performance Engineering), 31 (Memory Optimization),
                  32 (Memory-Mapped Arrays)
Progress:         32 / 75 chapters
Completed:        Ch1-29
Remaining:        43 chapters
```

Chapter 29 ended with "professional judgment about vectorization" — this trio gives you the
actual tools to exercise that judgment: how to profile properly (30), how to shrink memory
footprint deliberately (31), and how to work with datasets that don't fit in RAM at all (32).


In [ ]:
import numpy as np
import timeit
import time
print("NumPy version:", np.__version__)


---
# CHAPTER 30 — Performance Engineering

## Concept
"It feels slow" is not a diagnosis. Professional performance work means **measuring** with the
right tool, forming a hypothesis about the cause (CPU-bound compute? memory allocation?
non-vectorized fallback?), and verifying the fix actually helped — not guessing and rewriting.

## Why it matters
In production ML pipelines, the difference between a naive and a properly profiled
implementation is routinely 10-100x, translating directly into infrastructure cost and
iteration speed.


In [ ]:
# %timeit is the standard IN-NOTEBOOK benchmarking tool (multiple runs, reports best/mean +- std)
# Since this is a plain .py-style cell here, we use timeit.timeit / timeit.repeat directly --
# in an actual Jupyter session, prefer the %timeit magic for its nicer automatic repeat-count handling.

x = np.random.default_rng(0).normal(size=1_000_000)

def approach_a():
    return x ** 2

def approach_b():
    return np.multiply(x, x)

def approach_c():
    return np.square(x)

for name, fn in [("x ** 2", approach_a), ("np.multiply(x, x)", approach_b), ("np.square(x)", approach_c)]:
    times = timeit.repeat(fn, number=100, repeat=5)   # 5 independent repeats of 100 calls each
    best = min(times) / 100   # best-of-5 average per call -- reduces noise from other system activity
    print(f"{name:20s}: best-of-5 avg = {best*1e6:.2f} microseconds/call")


### Expected Output (verified on NumPy 2.4.4 — exact numbers vary by machine, the METHOD is what matters)
```
x ** 2              : best-of-5 avg = 707.48 microseconds/call
np.multiply(x, x)   : best-of-5 avg = 727.26 microseconds/call
np.square(x)        : best-of-5 avg = 681.63 microseconds/call
```
All three should land within noise of each other — they compile down to essentially the same
operation. This is a useful sanity-check pattern: when several "different" ways of writing
something benchmark identically, that's a hint they're the same underlying operation, and you
should pick whichever reads most clearly rather than agonizing over which is "faster."

### Why `timeit.repeat` (best-of-N), not a single `timeit.timeit` call
A single measurement is vulnerable to system noise (another process briefly using the CPU, a
garbage collection pause, thermal throttling). Taking the **minimum** across several independent
repeats is the standard robust benchmarking practice — it approximates the "clean run" time,
since noise can only ever make a run slower, never faster than its true cost. This is exactly
what Jupyter's `%timeit` magic does automatically.

### A structured performance-debugging workflow


In [ ]:
# --- A deliberately slow function, then a systematic diagnosis ---
def suspicious_function(data):
    # "Vectorized" but with a subtle issue: np.vectorize() does NOT compile to C --
    # it's essentially a Python loop in disguise, with extra overhead on top!
    vfunc = np.vectorize(lambda v: v * 2 + 1)
    return vfunc(data)

def properly_vectorized(data):
    return data * 2 + 1

data = np.random.default_rng(0).normal(size=200_000)

t_suspicious = timeit.timeit(lambda: suspicious_function(data), number=5)
t_proper = timeit.timeit(lambda: properly_vectorized(data), number=5)

print(f"np.vectorize() version   : {t_suspicious:.4f} s (5 calls)")
print(f"true vectorized version  : {t_proper:.4f} s (5 calls)")
print(f"np.vectorize() is {t_suspicious / t_proper:.0f}x SLOWER despite 'looking' vectorized")


### Expected Output (verified on NumPy 2.4.4 — np.vectorize should be dramatically slower)
```
np.vectorize() version   : 0.3270 s (5 calls)
true vectorized version  : 0.0029 s (5 calls)
np.vectorize() is 112x SLOWER despite 'looking' vectorized
```

### The `np.vectorize` trap — this is exactly the "diagnose, don't guess" lesson
`np.vectorize()` is a **convenience wrapper**, not a compiled vectorization mechanism — it
loops over elements in Python under the hood, calling the wrapped function once per element,
plus extra broadcasting/type-inference bookkeeping on every call. Code that "looks vectorized"
(no explicit `for` loop visible) can still be, functionally, a slow Python loop with more
overhead than a plain loop would have. **The only way to catch this is to actually benchmark
it** — reading the code doesn't reveal the problem; profiling does.

### Diagnostic checklist when a NumPy operation is "unexpectedly slow"
1. **Is it actually calling a compiled ufunc, or something like `np.vectorize`/`apply_along_axis`
   that hides a Python loop?** (Shown above.)
2. **Is the array non-contiguous**, forcing hidden internal copies (Chapter 28)?
   Check `.flags['C_CONTIGUOUS']`.
3. **Is memory the bottleneck, not compute?** Large temporary arrays (Chapter 29) can dominate
   runtime through allocation/deallocation and cache-thrashing, not the arithmetic itself.
4. **Is the dtype larger than necessary?** `float64` computations move twice the memory of
   `float32` for the same element count (Chapter 4, Chapter 31).
5. **Is this actually a single huge array operation, or thousands of tiny ones** — remember
   Chapter 29's lesson that per-call overhead dominates for small arrays; batching many tiny
   operations into fewer, larger ones can help enormously.

### Interview Insight
> "A colleague's NumPy code uses `np.vectorize()` and expects it to be fast because it's
> 'vectorized.' What would you tell them, and how would you verify your concern?"
`np.vectorize()` doesn't compile to C — it's a Python-level loop with extra dispatch overhead
per element, often *slower* than a plain `for` loop, and reliably slower than true array-native
vectorized operations. I'd verify with a direct `timeit` comparison against the equivalent
native NumPy expression (as done above) rather than relying on the name alone.


---
# CHAPTER 31 — Memory Optimization

## Concept
Every byte of dtype precision you don't need is memory (and often bandwidth/cache) wasted at
scale. Systematic memory optimization means choosing the smallest dtype that still safely
represents your data, avoiding unnecessary copies, and processing data in bounded chunks when
it doesn't comfortably fit in RAM.

## Why it matters
A 10-million-row dataset with 20 `float64` feature columns is `10e6 * 20 * 8 = 1.6 GB`; the same
data as `float32` is `800 MB`; as `float16` (where precision allows), `400 MB`. This directly
determines whether a dataset fits in RAM, in GPU memory, or needs to be processed in chunks at
all.


In [ ]:
n_rows, n_cols = 5_000_000, 10

for dt in [np.float64, np.float32, np.float16]:
    size_bytes = n_rows * n_cols * np.dtype(dt).itemsize
    print(f"{dt.__name__:10s}: {size_bytes / 1e6:8.1f} MB   ({np.dtype(dt).itemsize} bytes/element)")

print()
# Demonstrate a REAL precision loss from downcasting too aggressively
values = np.array([1.23456789, 100000.123456, 0.0000123456], dtype=np.float64)
print("float64:", values)
print("float32:", values.astype(np.float32), "  <- some precision lost, usually acceptable for ML")
print("float16:", values.astype(np.float16), "  <- SIGNIFICANT precision lost -- check if acceptable!")


### Expected Output (verified on NumPy 2.4.4)
```
float64  :   400.0 MB   (8 bytes/element)
float32  :   200.0 MB   (4 bytes/element)
float16  :   100.0 MB   (2 bytes/element)

float64: [1.23456789e+00 1.00000123e+05 1.23456000e-05]
float32: [1.23456788e+00 1.00000125e+05 1.23456002e-05]   <- tiny precision loss, usually acceptable for ML
float16: [1.234e+00           inf 1.234e-05]   <- 100000.12 OVERFLOWED to inf! float16's max is ~65504
```

### Reading the float16 result carefully
Notice `100000.123456` became **`inf`**, not just a rounded approximation — `float16`'s maximum
representable magnitude is only about `65504` (Chapter 4's range table), so a value like
`100000` doesn't just lose precision, it **overflows entirely** and silently becomes infinity
(with a `RuntimeWarning`, not a hard error). This is a more severe failure mode than the "some
precision lost" case for `float32` above, and it's exactly why **never downcast blindly** is the
rule: always check, for your actual data's range and required precision, whether the smaller
dtype can even represent your values at all, let alone precisely.

### Best Practice: a defensive downcast helper


In [ ]:
def safe_astype_report(arr, target_dtype, rtol=1e-3):
    """Cast to a smaller dtype and report the maximum relative error introduced."""
    original = arr.astype(np.float64)   # upcast for a fair, high-precision comparison baseline
    converted = arr.astype(target_dtype).astype(np.float64)
    with np.errstate(invalid='ignore', divide='ignore'):
        rel_error = np.abs(converted - original) / np.maximum(np.abs(original), 1e-12)
    max_rel_error = np.max(rel_error)
    ok = max_rel_error <= rtol
    print(f"Casting to {target_dtype.__name__}: max relative error = {max_rel_error:.6f}   "
          f"{'OK' if ok else 'EXCEEDS rtol=' + str(rtol) + ' -- reconsider!'}")
    return ok

data = np.random.default_rng(0).uniform(0, 1000, size=1000)
safe_astype_report(data, np.float32)
safe_astype_report(data, np.float16)


### Expected Output (verified on NumPy 2.4.4)
```
Casting to float32: max relative error = 0.000000   OK
Casting to float16: max relative error = 0.000471   OK
```
For *this* particular dataset (uniform values between 0 and 1000, well within `float16`'s
range), even `float16` stays under the `rtol=1e-3` threshold — a reminder that whether a
downcast is "safe" depends entirely on your actual data's range and distribution, not on the
dtype name alone. Contrast this with the `100000.12` value two cells above, which overflowed
`float16` completely — the *same* target dtype was catastrophic for one dataset and perfectly
fine for another. This is precisely why `safe_astype_report`-style checks belong in a pipeline
as a standing guard, not a one-time manual judgment call.

### `astype(copy=False)` — avoiding an unnecessary copy when dtype already matches
```python
X.astype(np.float32, copy=False)   # returns X unchanged (no copy) if X is ALREADY float32,
                                     # otherwise converts (necessarily copying)
```
This mirrors the `asarray` lesson from Chapter 2 — avoid paying for a copy you don't need,
especially inside functions called repeatedly on data that may already be in the target dtype.

### Chunked processing for datasets that don't comfortably fit in memory


In [ ]:
def process_in_chunks(total_size, chunk_size, process_fn):
    """Process a conceptually huge dataset in bounded-memory chunks, accumulating a result."""
    total = 0.0
    for start in range(0, total_size, chunk_size):
        end = min(start + chunk_size, total_size)
        # In a real pipeline, this chunk would be READ FROM DISK here, not generated in memory --
        # simulating that by generating one chunk at a time instead of one giant array up front.
        chunk = np.random.default_rng(start).normal(size=end - start).astype(np.float32)
        total += process_fn(chunk)
    return total

result = process_in_chunks(
    total_size=50_000_000,     # simulate 50 million values -- 200MB as float32, but imagine 50GB
    chunk_size=1_000_000,       # process 1 million at a time -- bounded, small memory footprint
    process_fn=lambda c: np.sum(c ** 2)
)
print(f"Sum of squares over 50,000,000 simulated values: {result:.2f}")
print("Peak memory for this computation stayed bounded near ONE chunk's size (~4MB),")
print("regardless of how large `total_size` conceptually is.")


### Expected Output (verified on NumPy 2.4.4)
```
Sum of squares over 50,000,000 simulated values: 49980364.00
Peak memory for this computation stayed bounded near ONE chunk's size (~4MB),
regardless of how large `total_size` conceptually is.
```

### Common Mistake
Loading an entire large dataset into memory "to be safe," then discovering only at scale that
it doesn't fit — chunked processing patterns (read a bounded piece, process, accumulate,
discard, repeat) should be the default design for any pipeline whose input size isn't tightly
bounded in advance, not a reactive fix applied after a `MemoryError`.

### Interview Insight
> "How would you process a 100GB numerical dataset on a machine with 16GB RAM?"
Process it in bounded-size chunks read from disk (or a memory-mapped array, Chapter 32), using
the smallest dtype that safely represents the data's actual required precision, accumulating
only the final result (or writing intermediate results back to disk) rather than ever holding
the full dataset in memory at once.


---
# CHAPTER 32 — Memory-Mapped Arrays

## Concept
`np.memmap` creates an array-like object backed by a file **on disk**, not in RAM. Data is read
into memory only as/when actually accessed (via the operating system's virtual memory /
paging mechanism), letting you work with arrays far larger than available RAM using ordinary
NumPy-like indexing syntax.

## Why it matters
This is the direct, practical answer to "my dataset is bigger than my RAM" — genuinely bigger,
not just "inconveniently large" (which Chapter 31's chunking already handles). Memmap is
standard in genomics, satellite imagery, and any domain with arrays in the tens-to-hundreds of
GB range.


In [ ]:
import os, tempfile

# Create a memory-mapped array backed by a temp file on disk
shape = (10_000, 1000)   # 10M elements
tmp_path = os.path.join(tempfile.gettempdir(), "numpy_mastery_demo.dat")

mmap_arr = np.memmap(tmp_path, dtype=np.float32, mode='w+', shape=shape)
print("Created memmap array, shape:", mmap_arr.shape, " dtype:", mmap_arr.dtype)
print("Type:", type(mmap_arr))
print("File size on disk:", os.path.getsize(tmp_path) / 1e6, "MB")

# Write to it just like a normal array -- writes go to the underlying FILE, not just RAM
mmap_arr[0, :] = np.arange(1000)
mmap_arr[5000, :5] = [1, 2, 3, 4, 5]
mmap_arr.flush()   # ensure writes are committed to disk

print("\nmmap_arr[0, :5] :", mmap_arr[0, :5])
print("mmap_arr[5000, :5]:", mmap_arr[5000, :5])


### Expected Output
```
Created memmap array, shape: (10000, 1000)  dtype: float32
Type: <class 'numpy.memmap'>
File size on disk: 40.0 MB

mmap_arr[0, :5] : [0. 1. 2. 3. 4.]
mmap_arr[5000, :5]: [1. 2. 3. 4. 5.]
```

### The core idea: it behaves like a normal ndarray, but isn't fully resident in RAM
`mmap_arr[0, :]` triggers the OS to page in just the memory needed for that slice from disk; the
rest of the 40MB file stays on disk until (if ever) accessed. For a genuinely huge file (say,
200GB) on a machine with 16GB RAM, this means you can still do `huge_arr[some_index]` and get
correct results — the OS handles paging transparently — as long as your *access pattern* touches
a bounded working set at a time (exactly the chunking philosophy from Chapter 31, but now
enforced by the file itself rather than your own loop).

### Reading an existing memmap (mode='r') for read-only access


In [ ]:
# Re-open the SAME file, this time read-only, simulating a separate process/session reading it
mmap_readonly = np.memmap(tmp_path, dtype=np.float32, mode='r', shape=shape)
print("Read back mmap_readonly[0, :5] :", mmap_readonly[0, :5])
print("Read back mmap_readonly[5000, :5]:", mmap_readonly[5000, :5])

try:
    mmap_readonly[0, 0] = 999   # attempting to write to a read-only memmap
except ValueError as e:
    print("\nAttempting to write raised ValueError:", e)

# Clean up the demo file
del mmap_arr, mmap_readonly
os.remove(tmp_path)
print("\nCleaned up demo file.")


### Expected Output
```
Read back mmap_readonly[0, :5] : [0. 1. 2. 3. 4.]
Read back mmap_readonly[5000, :5]: [1. 2. 3. 4. 5.]

Attempting to write raised ValueError: assignment destination is read-only

Cleaned up demo file.
```

### `mode` parameter — matters a lot
| mode | Meaning |
|---|---|
| `'r'` | read-only; file must already exist |
| `'r+'` | read-write; file must already exist |
| `'w+'` | create (or overwrite!) the file, read-write |
| `'c'` | copy-on-write — reads see the file, writes go to an in-memory copy that's discarded, file is never modified |

`'c'` mode is particularly useful for **experimentation on huge read-only source data**: you can
freely "modify" a memmap'd array in your session for testing without any risk of corrupting the
underlying (possibly very expensive to regenerate) source file.

### Real-world ML pipeline pattern
```python
# One-time preprocessing step, run once, writes a huge feature array to disk:
features = np.memmap('features.dat', dtype=np.float32, mode='w+', shape=(50_000_000, 128))
for chunk_start in range(0, 50_000_000, 100_000):
    # ... compute a chunk of features ...
    features[chunk_start:chunk_start+100_000] = chunk_result
features.flush()

# Later, in a DIFFERENT training run/process, re-open read-only, use like a normal array:
features = np.memmap('features.dat', dtype=np.float32, mode='r', shape=(50_000_000, 128))
batch = features[batch_indices]   # only THIS batch's data is actually read from disk
```
This pattern — write once with chunked computation (Chapter 31), read many times via memmap
with only the needed slices actually loaded — is standard for large ML training datasets that
exceed available RAM.

### Common Mistake
Using `mode='w+'` on a file path that already contains real data you didn't intend to overwrite
— `'w+'` truncates/creates, exactly like opening a file with `'w'` in plain Python. Always
double-check `mode` matches intent (`'r+'` to modify existing data, `'r'` for pure read access).

### Interview Insight
> "How would you work with a 500GB numerical array on a machine with 32GB RAM, using NumPy?"
`np.memmap` — it presents ordinary array-indexing syntax backed by a disk file rather than RAM,
letting the OS page in only the data actually accessed. Combine with chunked, bounded-working-set
access patterns (Chapter 31) so you never need the full array resident in memory at once, and
choose `mode` (`'r'`, `'r+'`, `'w+'`, `'c'`) deliberately based on whether you need read-only,
in-place-modify, create, or copy-on-write semantics.


---
## Chapters 30–32 Summary

- **Performance engineering**: measure with `timeit`/`%timeit` (best-of-N, not a single run) —
  never guess. `np.vectorize()` is a documented trap: it "looks" vectorized but is actually a
  slow Python loop with extra overhead, verifiable only by benchmarking, not by reading the code.
- **Memory optimization**: dtype choice directly and linearly determines memory footprint
  (`float64` → `float32` → `float16` halves each time); always verify precision loss is
  acceptable for your actual data range before downcasting; chunked processing keeps peak
  memory bounded regardless of total logical dataset size.
- **Memory-mapped arrays**: `np.memmap` provides ordinary NumPy indexing syntax over disk-backed
  data, letting the OS page in only what's accessed — the standard tool for arrays genuinely
  larger than available RAM, with `mode` (`r`/`r+`/`w+`/`c`) controlling read/write/create/
  copy-on-write semantics.

---

## Mastery Exercises

**Exercise 1:** A teammate claims their preprocessing function is "already vectorized" because
it uses `np.vectorize()`. What single benchmarking step would you ask them to run to verify (or
refute) this claim?

**Exercise 2 (Prediction):** A `(2_000_000, 15)` feature matrix currently stored as `float64` is
causing memory pressure. Compute the memory saved (in MB) by converting to `float32`, and state
one concrete check you'd run before doing so in production.

**Coding Challenge:** Implement `benchmark_dtype_impact(n, dtype_list)` that creates a random
array of size `n` in each given dtype, times a representative operation (e.g., `arr.sum()`)
using `timeit.repeat` (best-of-5), and returns a dict mapping dtype name to (memory_MB,
best_time_seconds). Use it to empirically verify whether `float32` is meaningfully faster than
`float64` for a large-enough `n` on your machine, not just smaller in memory.

**Debugging Challenge:**
```python
big_features = np.memmap('features.dat', dtype=np.float32, mode='w+', shape=(1_000_000, 50))
# ... six months later, a teammate re-runs this SAME line to "load" the file for training ...
# and discovers all their carefully computed features are now all zeros!
```
Explain exactly what went wrong (referencing the `mode` parameter table), and the one-word fix.

**Interview Questions:**
1. Why can code "look" vectorized (no visible `for` loop) and still perform like a slow Python
   loop? Give the specific NumPy function most responsible for this trap.
2. What's the actual, mechanical reason `timeit` results should use the minimum of several
   repeats rather than a single measurement or an average?
3. Give two concrete risks of downcasting a `float64` array to `float16` without checking the
   data first.
4. How does `np.memmap` let you work with an array larger than available RAM, at a mechanical
   level (what's actually happening when you index into it)?

**Quiz:**
- Q1: True/False — `np.vectorize()` compiles the wrapped Python function to C for genuine
  vectorized execution.
- Q2: Converting a `(1_000_000, 20)` `float64` array to `float32` reduces its memory footprint
  by what factor?
- Q3: Which `memmap` mode would you use to experiment with modifications to a huge read-only
  source file without risking corrupting it?

---


